<a href="https://colab.research.google.com/github/rania10082004/Syntecxhub_Spam_Detection/blob/main/Spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import pandas as pd
import numpy as np
import warnings
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

In [2]:
# 1. DATA LOADING
path = kagglehub.dataset_download('balaka18/email-spam-classification-dataset-csv')
df = pd.read_csv(f"{path}/emails.csv")

100%|██████████| 1.66M/1.66M [00:00<00:00, 2.96MB/s]

Extracting files...


In [3]:
# 2. DATA PREPARATION
X = df.drop(['Email No.', 'Prediction'], axis=1)
y = df['Prediction']
model_vocabulary = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=26)

In [4]:
# 3. MODEL TRAINING (Both Models)
# Logistic Regression
lr_model = LogisticRegression(max_iter=500)
lr_model.fit(X_train, y_train)

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Print Accuracies to Console
lr_acc = accuracy_score(y_test, lr_model.predict(X_test))
nb_acc = accuracy_score(y_test, nb_model.predict(X_test))
print(f"Logistic Regression Accuracy: {lr_acc:.2%}")
print(f"Naive Bayes Accuracy: {nb_acc:.2%}")

Logistic Regression Accuracy: 97.10%
Naive Bayes Accuracy: 93.43%


In [5]:
# 4. PREDICTION FUNCTION FOR GRADIO
def compare_models(email_content):
    if not email_content.strip():
        return "Empty Input", "Empty Input"

    # Pre-processing: Map input text to the 3000-word feature set
    words_in_input = email_content.lower().split()
    input_data = {word: [0] for word in model_vocabulary}

    for word in words_in_input:
        if word in input_data:
            input_data[word][0] += 1

    input_df = pd.DataFrame(input_data)[model_vocabulary]

    # Logistic Regression Prediction
    lr_pred = lr_model.predict(input_df)[0]
    lr_prob = lr_model.predict_proba(input_df)[0]
    lr_result = f"{'🚨 SPAM' if lr_pred == 1 else '✅ NOT SPAM'}\n({max(lr_prob)*100:.1f}% confidence)"

    # Naive Bayes Prediction
    nb_pred = nb_model.predict(input_df)[0]
    nb_prob = nb_model.predict_proba(input_df)[0]
    nb_result = f"{'🚨 SPAM' if nb_pred == 1 else '✅ NOT SPAM'}\n({max(nb_prob)*100:.1f}% confidence)"

    return lr_result, nb_result

In [6]:
# 5. GRADIO INTERFACE DESIGN
demo = gr.Interface(
    fn=compare_models,
    inputs=gr.Textbox(
        lines=7,
        placeholder="Paste  content here...",
        label="Message"
    ),
    outputs=[
        gr.Textbox(label="Logistic Regression Result"),
        gr.Textbox(label="Naive Bayes Result")
    ],
    title="📧 Spam Detection: Model Comparison",
    description="Compare how Logistic Regression and Naive Bayes to classify based on word frequencies.",
    theme="huggingface",
    examples=[
        ["WINNER! You have been selected for a cash prize of $5000. Click the link to claim."],
        ["Hi, I've attached the project report for your review. Let me know if you have questions."]
    ]
)



In [7]:
# 6. LAUNCH
if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23d4b03627448a9bdc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
